## CNN-RNN + XGBoost

In [ ]:
%pip install pandas
%pip install scikit-learn
%pip install tensorflow
%pip install matplotlib
%pip install imbalanced-learn
%env SCIPY_ARRAY_API=1
%pip install xgboost lightgbm numpy

In [ ]:
# 📦 Imports
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.data_utils import load_and_preprocess_data
from src.model import build_weighted_model  # make sure this is imported
import csv
import pandas as pd
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import recall_score
from sklearn.metrics import precision_recall_curve
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score
import joblib


In [ ]:
# 1. Load and preprocess the data
(X_train, X_test, y_train, y_test), scaler = load_and_preprocess_data("../data/dataset.csv")
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [ ]:
# ✅ 2. Load trained CNN-RNN model and get predictions
from tensorflow.keras.models import load_model
model = load_model("../results/cnn_rnn_model_dataset.keras")
cnn_train_probs = model.predict(X_train).flatten()
cnn_test_probs = model.predict(X_test).flatten()

In [ ]:
# ✅ 3. Train XGBoost on flat input

from xgboost import XGBClassifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train_flat, y_train)
xgb_train_probs = xgb.predict_proba(X_train_flat)[:, 1]
xgb_test_probs = xgb.predict_proba(X_test_flat)[:, 1]

In [ ]:
# ✅ 4. Create meta-features for logistic regression
meta_X_train = np.vstack([cnn_train_probs, xgb_train_probs]).T
meta_X_test = np.vstack([cnn_test_probs, xgb_test_probs]).T

In [ ]:
# ✅ 5. Train logistic regression as meta-learner
from sklearn.linear_model import LogisticRegression
meta_learner = LogisticRegression()
meta_learner.fit(meta_X_train, y_train)
ensemble_probs = meta_learner.predict_proba(meta_X_test)[:, 1]

In [ ]:
# ✅ 6. Final prediction and evaluation
threshold = 0.36  # Same threshold as before
y_pred = (ensemble_probs > threshold).astype(int)

print("\n📊 Ensemble Classification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix - Stacked Ensemble")
plt.grid(False)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Define number of samples	
n_samples = 2000

# Simulated data ranges
np.random.seed(42)
data = {
    'q': np.random.uniform(-3.14, 3.14, n_samples),
    'dq': np.random.uniform(-5.0, 5.0, n_samples),
    'ddq': np.random.uniform(-10.0, 10.0, n_samples),
    'tau_est': np.random.uniform(-50.0, 50.0, n_samples),
    'temperature_motor': np.random.randint(20, 70, n_samples),
    'gyro_x': np.random.normal(0, 0.05, n_samples),
    'gyro_y': np.random.normal(0, 0.05, n_samples),
    'gyro_z': np.random.normal(0, 0.05, n_samples),
    'accel_x': np.random.normal(0, 1.0, n_samples),
    'accel_y': np.random.normal(0, 1.0, n_samples),
    'accel_z': np.random.normal(-9.8, 1.0, n_samples),
    'rpy_roll': np.random.uniform(-180, 180, n_samples),
    'rpy_pitch': np.random.uniform(-90, 90, n_samples),
    'rpy_yaw': np.random.uniform(-180, 180, n_samples),
    'temperature_imu': np.random.randint(20, 60, n_samples),
    'soc': np.random.randint(0, 100, n_samples),
    'current': np.random.uniform(-30, 30, n_samples),
    'cell_vol_0': np.random.uniform(3.2, 4.2, n_samples),
    'cell_vol_1': np.random.uniform(3.2, 4.2, n_samples),
    'power_v': np.random.uniform(20, 28, n_samples),
    'power_a': np.random.uniform(0, 10, n_samples),
    'fan_freq_0': np.random.randint(0, 2000, n_samples),
    'fan_freq_1': np.random.randint(0, 2000, n_samples),
    'machine_failure': np.random.randint(0, 2, n_samples)
}

# Create and save DataFrame
df = pd.DataFrame(data)
df.to_csv("robot_sensor_data.csv", index=False)
print("✅ robot_sensor_data.csv created with", df.shape[0], "rows and", df.shape[1], "columns")
